# P40 demo

We generate a few prompts from our set with SD 1.5 and SDXL on the same seed and ask Qwen2.5-VL about every constraint. On a T4: run all, then run all again after the automatic restart.

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = Path("/content/cv_project")
if not (REPO / "pyproject.toml").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/IzyumovMaxim/cv_project", str(REPO)], check=True)
os.chdir(REPO)
sys.path[:0] = [str(REPO / "src"), str(REPO / "scripts")]

MARKER = Path("/content/.demo_installed")
if not MARKER.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "bitsandbytes==0.47.0"], check=True)
    MARKER.touch()
    print("Installed. Restarting the session; run all cells again.")
    os.kill(os.getpid(), 9)
print("Commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
GPU = torch.cuda.get_device_name(0)
print(GPU, round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")

## 2. Prompts

Three prompts from the factorial set: a green horse, the same horse with a black book, and three blue metal benches on the left.

In [ ]:
import hashlib
import pandas as pd
from attribute_interference.prompts import build
from attribute_interference.storage import atomic_json, digest, load_config, source_hash, write_csv

CONFIGS = {"SD 1.5": "configs/main_sd15.yaml", "SDXL": "configs/main_sdxl.yaml"}
PROMPTS, PAIRS = build(load_config(CONFIGS["SD 1.5"]))
BY_ID = {p["prompt_id"]: p for p in PROMPTS}
print(f"{len(PROMPTS)} prompts, {len(PAIRS)} minimal pairs")

DEMO_JOBS = [
    ("e9e226eda3b2fff5", 1),  # a green horse
    ("a0870a216985d85a", 1),  # a green horse and a black book
    ("cdc2468937782175", 2),  # three blue metal benches on the left
]
pd.DataFrame([{"image_id": f"{pid}_s{seed}", "constraints": BY_ID[pid]["active_factors"],
               "prompt": BY_ID[pid]["prompt"]} for pid, seed in DEMO_JOBS])

## 3. Demo runs

The same run setup as `prepare`, but only for these three images.

In [ ]:
DEMO_ROOT = Path("/content/demo_runs")


def make_run(name, config, jobs):
    root = DEMO_ROOT / name
    if (root / "run.json").exists():
        return root
    cfg = load_config(config)
    ids = {pid for pid, _ in jobs}
    root.mkdir(parents=True, exist_ok=True)
    write_csv(root / "prompts.csv", [p for p in PROMPTS if p["prompt_id"] in ids])
    write_csv(root / "pairs.csv", [p for p in PAIRS if p["base_id"] in ids and p["added_id"] in ids],
              ["base_id", "added_id", "added_factor"])
    write_csv(root / "jobs.csv", [{"image_id": f"{pid}_s{seed}", "prompt_id": pid, "seed": seed} for pid, seed in jobs])
    atomic_json(root / "run.json", {
        "config": cfg, "config_hash": digest(cfg), "source_hash": source_hash(),
        "prompt_count": len(ids), "job_count": len(jobs),
        "data_hashes": {n: hashlib.sha256((root / n).read_bytes()).hexdigest()
                        for n in ["prompts.csv", "pairs.csv", "jobs.csv"]}})
    return root


RUNS = {model: make_run(model.replace(" ", "").replace(".", "").lower(), cfg, DEMO_JOBS)
        for model, cfg in CONFIGS.items()}
RUNS

## 4. Generation

In [ ]:
!attribute-interference generate --config {CONFIGS["SD 1.5"]} --output {RUNS["SD 1.5"]}

SDXL is loaded in fp16, as in `sdxl_generation_evaluation.ipynb`, to fit in Colab memory.

In [ ]:
GEN_SDXL = Path("/content/gen_sdxl_lowmem.py")
GEN_SDXL.write_text("""import sys, gc
import attribute_interference.storage as st
import attribute_interference.generation as gen
from diffusers import StableDiffusionXLPipeline

_from_pretrained = StableDiffusionXLPipeline.from_pretrained

@classmethod
def lowmem_from_pretrained(cls, *args, **kwargs):
    kwargs.setdefault("variant", "fp16")
    kwargs.setdefault("low_cpu_mem_usage", True)
    pipe = _from_pretrained(*args, **kwargs)
    gc.collect()
    return pipe

StableDiffusionXLPipeline.from_pretrained = lowmem_from_pretrained
gen.generate(st.load_config(sys.argv[1]), sys.argv[2])
""")
!PYTHONPATH=src python {GEN_SDXL} {CONFIGS["SDXL"]} {RUNS["SDXL"]}

## 5. Labeling

Qwen2.5-VL in 8-bit answers one question per constraint; OWLv2 counts objects for comparison.

In [ ]:
!PYTHONPATH=src python scripts/evaluate_labels.py --config {CONFIGS["SD 1.5"]} --run {RUNS["SD 1.5"]} --limit 100

In [ ]:
!PYTHONPATH=src python scripts/evaluate_labels.py --config {CONFIGS["SDXL"]} --run {RUNS["SDXL"]} --limit 100

## 6. Model outputs

In [ ]:
import json
import matplotlib.pyplot as plt
from PIL import Image

NAMES = {"object": "Object", "C": "Color", "N": "Count", "P": "Position", "M": "Material", "O2": "Second object"}


def qwen_record(model, image_id):
    return json.loads((RUNS[model] / "qwen_int8_records" / f"{image_id}.json").read_text())


def generation_record(model, image_id):
    return json.loads((RUNS[model] / "generation_records" / f"{image_id}.json").read_text())


def answers(model, image_id):
    rec = qwen_record(model, image_id)
    rows = {}
    for factor, raw in rec["raw"].items():
        label = rec["parsed"].get(factor)
        if rec["parsed"].get("object") is False and factor != "object":
            label = False
        rows[factor] = (raw["answer"].strip(), label)
    return rows


def mark(label):
    return {True: "✓", False: "✗"}.get(label, "?")


def strict(rows):
    labels = [label for _, label in rows.values()]
    return None if None in labels else all(labels)


def show(pid, seed):
    image_id = f"{pid}_s{seed}"
    p = BY_ID[pid]
    fig, axes = plt.subplots(1, 2, figsize=(9, 4.6))
    table = {}
    for ax, model in zip(axes, CONFIGS):
        gen = generation_record(model, image_id)
        ax.imshow(Image.open(RUNS[model] / gen["path"]))
        ax.set_axis_off()
        rows = answers(model, image_id)
        ax.set_title(f"{model}: strict {mark(strict(rows))}  ({gen['seconds']:.1f} s)", fontsize=11)
        for factor, (answer, label) in rows.items():
            table.setdefault(NAMES[factor], {})[f"{model} answer"] = answer
            table[NAMES[factor]][f"{model}"] = mark(label)
    fig.suptitle(f'"{p["prompt"]}", seed {seed}', fontsize=12)
    plt.tight_layout()
    plt.show()
    questions = {NAMES[f]: r["question"] for f, r in qwen_record("SD 1.5", image_id)["raw"].items()}
    df = pd.DataFrame(table).T
    df.insert(0, "question", pd.Series(questions))
    display(df)

In [ ]:
show("e9e226eda3b2fff5", 1)

In [ ]:
show("a0870a216985d85a", 1)

In [ ]:
show("cdc2468937782175", 2)